# **Tool Calling in LangChain**

In [ ]:
import os
os.environ["GROQ_API_KEY"] = "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"

In [635]:
from langchain_groq import ChatGroq
from langchain.tools import tool
from langchain_core.messages import HumanMessage
import requests

## **1. Tool Creation**

In [636]:
@tool

def multiply(a:int,b:int) ->int:
    """Given 2 numbers a and b, this tool returns their product"""
    return a * b

In [637]:
print(multiply.invoke({"a":3,"b":4}))

12


In [638]:
print(multiply.name)
print(multiply.description)
print(multiply.args)

multiply
Given 2 numbers a and b, this tool returns their product
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


## **2. Tool Binding**

In [639]:
llm = ChatGroq(model_name="llama-3.1-8b-instant")

In [640]:
llm_with_tools = llm.bind_tools([multiply])

In [641]:
query = HumanMessage("Can you multiply 3 with 10")
messages = [query]
messages

[HumanMessage(content='Can you multiply 3 with 10', additional_kwargs={}, response_metadata={})]

In [642]:
result = llm_with_tools.invoke(messages)
result

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'vw8f30n0t', 'function': {'arguments': '{"a":3,"b":10}', 'name': 'multiply'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 240, 'total_tokens': 259, 'completion_time': 0.029417019, 'completion_tokens_details': None, 'prompt_time': 0.015906121, 'prompt_tokens_details': None, 'queue_time': 0.046325113, 'total_time': 0.04532314}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_f757f4b0bf', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d48da-29ec-7450-97b7-e4298f41d2f7-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 10}, 'id': 'vw8f30n0t', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 240, 'output_tokens': 19, 'total_tokens': 259})

In [643]:
messages.append(result)

In [644]:
messages

[HumanMessage(content='Can you multiply 3 with 10', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'vw8f30n0t', 'function': {'arguments': '{"a":3,"b":10}', 'name': 'multiply'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 240, 'total_tokens': 259, 'completion_time': 0.029417019, 'completion_tokens_details': None, 'prompt_time': 0.015906121, 'prompt_tokens_details': None, 'queue_time': 0.046325113, 'total_time': 0.04532314}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_f757f4b0bf', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d48da-29ec-7450-97b7-e4298f41d2f7-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 10}, 'id': 'vw8f30n0t', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 240, 'output_tokens': 19, 'total_tokens': 259})]

In [645]:
tool_result = multiply.invoke(result.tool_calls[0])

In [646]:
tool_result.content

'30'

In [647]:
messages.append(tool_result)

In [648]:
messages

[HumanMessage(content='Can you multiply 3 with 10', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'vw8f30n0t', 'function': {'arguments': '{"a":3,"b":10}', 'name': 'multiply'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 240, 'total_tokens': 259, 'completion_time': 0.029417019, 'completion_tokens_details': None, 'prompt_time': 0.015906121, 'prompt_tokens_details': None, 'queue_time': 0.046325113, 'total_time': 0.04532314}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_f757f4b0bf', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d48da-29ec-7450-97b7-e4298f41d2f7-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 10}, 'id': 'vw8f30n0t', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 240, 'output_tokens': 19, 'total_tokens': 259}),
 ToolMessage(c

In [649]:
# Finally
llm_with_tools.invoke(messages).content

"There's no need to call another function as the answer is already clear."

## **3. Currency Converter Tool**

In [650]:
from langchain_core.tools import InjectedToolArg
from typing import Annotated
@tool
def get_conversion_fector(base_currency:str, target_currency:str) ->float:
    """
    This function fetches the currency conversion fector between the base and target currencies
    """
    url = f"https://v6.exchangerate-api.com/v6/7e33c4d15f697bbf3ced0ed5/pair/{base_currency}/{target_currency}"

    response = requests.get(url=url)
    return response.json()


@tool

def convert(base_currency_value:int,conversion_rate: Annotated[float,InjectedToolArg]) ->float:
    """
    Given a conversion currency value this function calculates the target currency value from a base currency
    value
    """
    return base_currency_value * conversion_rate

In [651]:
get_conversion_fector.invoke({"base_currency":"USD","target_currency":"PKR"})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1775001601,
 'time_last_update_utc': 'Wed, 01 Apr 2026 00:00:01 +0000',
 'time_next_update_unix': 1775088001,
 'time_next_update_utc': 'Thu, 02 Apr 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'PKR',
 'conversion_rate': 279.0907}

In [652]:
convert.invoke({"base_currency_value":10,"conversion_rate":279.0907})

2790.907

In [653]:
llm = ChatGroq(model_name="llama-3.1-8b-instant")

In [654]:
llm_with_tools_1 = llm.bind_tools([get_conversion_fector,convert])

In [655]:
query = "What is the conversion fector between the USD and PKR and based on that can you convert 10 USD to PKR"
messages = [HumanMessage(query)]
messages

[HumanMessage(content='What is the conversion fector between the USD and PKR and based on that can you convert 10 USD to PKR', additional_kwargs={}, response_metadata={})]

In [656]:
ai_message = llm_with_tools_1.invoke(messages)
ai_message.tool_calls

[{'name': 'get_conversion_fector',
  'args': {'base_currency': 'USD', 'target_currency': 'PKR'},
  'id': 'b5hw3famn',
  'type': 'tool_call'},
 {'name': 'convert',
  'args': {'base_currency_value': 10,
   'conversion_fector': 0.0158,
   'target_currency': 'PKR'},
  'id': '0w90446ef',
  'type': 'tool_call'}]

In [657]:
import json
for tool_call in ai_message.tool_calls:

    # -------------------- execute the first tool to get the conversion rate --------------------
    if tool_call['name']  == "get_conversion_fector":
        tool_message1 = get_conversion_fector.invoke(tool_call)
        # 1st Fetch the convertion rate
        conversion_rate = json.loads(tool_message1.content)['conversion_rate']
        # 2nd append this tool message into message list
        messages.append(tool_message1)

    # -------------------- execute the 2nd tool using the conversion rate of tool 1 --------------------

    if tool_call["name"] == "convert":
        # 1st fetch the current arg
        tool_call['args']['conversion_rate'] = conversion_rate
        tool_message2 = convert.invoke(tool_call)
        # 2nd append it also in messages list
        messages.append(tool_message2)

In [661]:
messages

[HumanMessage(content='What is the conversion fector between the USD and PKR and based on that can you convert 10 USD to PKR', additional_kwargs={}, response_metadata={}),
 ToolMessage(content='{"result": "success", "documentation": "https://www.exchangerate-api.com/docs", "terms_of_use": "https://www.exchangerate-api.com/terms", "time_last_update_unix": 1775001601, "time_last_update_utc": "Wed, 01 Apr 2026 00:00:01 +0000", "time_next_update_unix": 1775088001, "time_next_update_utc": "Thu, 02 Apr 2026 00:00:01 +0000", "base_code": "USD", "target_code": "PKR", "conversion_rate": 279.0907}', name='get_conversion_fector', tool_call_id='b5hw3famn'),
 ToolMessage(content='2790.907', name='convert', tool_call_id='0w90446ef')]

In [659]:
response = llm_with_tools_1.invoke(messages)
print("Full response:", response)
print("Content:", response.content)

Full response: content='' additional_kwargs={'tool_calls': [{'id': 'dj6jcyga3', 'function': {'arguments': '{"base_currency":"USD","target_currency":"PKR"}', 'name': 'get_conversion_fector'}, 'type': 'function'}, {'id': 'ew5zdg6ea', 'function': {'arguments': '{"base_currency_value":10,"conversion_rate":279.0907}', 'name': 'convert'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 50, 'prompt_tokens': 499, 'total_tokens': 549, 'completion_time': 0.07263503, 'completion_tokens_details': None, 'prompt_time': 0.031648624, 'prompt_tokens_details': None, 'queue_time': 0.045032275, 'total_time': 0.104283654}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019d48da-461c-79e0-ad9b-97d4581429c7-0' tool_calls=[{'name': 'get_conversion_fector', 'args': {'base_currency': 'USD', 'target_currency': 'PKR'}, 'id': 'dj6jcyga3', 't